# Pipeline 4 - SVM calibrado dual-head com 40.000 artigos

Treina dois modelos sobre os mesmos embeddings SPECTER:

1. multiclass para a categoria primaria;
2. One-vs-Rest multi-label para todas as categorias aplicaveis.

A ordem e avaliada apenas na categoria primaria. Os rotulos secundarios sao
avaliados como conjunto. O notebook exporta um unico artefato `.joblib` para o backend.


In [ ]:
!pip -q install iterative-stratification

from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, confusion_matrix, coverage_error,
    f1_score, hamming_loss, label_ranking_average_precision_score,
    log_loss, precision_recall_fscore_support, precision_score, recall_score,
    top_k_accuracy_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0]

TARGET_CATS = [
    'cs.LG', 'cs.AI', 'cs.CL', 'cs.CV', 'hep-ph', 'hep-th', 'gr-qc',
    'quant-ph', 'astro-ph', 'math-ph', 'math.MP', 'cond-mat.mtrl-sci',
    'cond-mat.mes-hall', 'cond-mat.str-el', 'cond-mat.stat-mech'
]


## Carregamento dos dados


In [ ]:
BASE = Path('/content/drive/MyDrive/projetoIA-EquipeLoremIpsum (1)')
PIPE = BASE / 'pipelines'
NOME_JSONL = 'arxiv_amostra_40000_com_embeddings_multilabel.json'
JSONL = PIPE / NOME_JSONL
if not JSONL.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {JSONL}')

df = pd.read_json(JSONL, lines=True)
X = np.vstack(df['embedding'].to_numpy()).astype(np.float32)

if 'target_categories' not in df.columns:
    df['target_categories'] = df['categories'].fillna('').map(
        lambda value: [c for c in str(value).split() if c in TARGET_CATS]
    )

primary_encoder = LabelEncoder().fit(TARGET_CATS)
y_primary = primary_encoder.transform(df['primary_category'])
mlb = MultiLabelBinarizer(classes=TARGET_CATS)
Y_multi = mlb.fit_transform(df['target_categories'])

print('X:', X.shape)
print('Primarias:', dict(zip(*np.unique(df['primary_category'], return_counts=True))))
print('Cardinalidade media:', Y_multi.sum(axis=1).mean())


## Split 75/10/15 com estratificacao multi-label iterativa


In [ ]:
primary_one_hot = np.eye(len(primary_encoder.classes_), dtype=np.int8)[y_primary]
Y_strat = np.hstack([Y_multi, primary_one_hot])

split_75_25 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
idx_train, idx_temp = next(split_75_25.split(X, Y_strat))

split_10_15 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.60, random_state=SEED)
idx_val_rel, idx_test_rel = next(split_10_15.split(X[idx_temp], Y_strat[idx_temp]))
idx_val = idx_temp[idx_val_rel]
idx_test = idx_temp[idx_test_rel]

X_train, X_val, X_test = X[idx_train], X[idx_val], X[idx_test]
yp_train, yp_val, yp_test = y_primary[idx_train], y_primary[idx_val], y_primary[idx_test]
Ym_train, Ym_val, Ym_test = Y_multi[idx_train], Y_multi[idx_val], Y_multi[idx_test]

print('Treino:', X_train.shape, 'Validacao:', X_val.shape, 'Teste:', X_test.shape)
print('Proporcoes:', len(idx_train)/len(X), len(idx_val)/len(X), len(idx_test)/len(X))


## Distribuicao de classes


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
pd.Series(primary_encoder.inverse_transform(yp_train)).value_counts().reindex(TARGET_CATS).plot.bar(ax=axes[0])
axes[0].set_title('Distribuicao natural das categorias primarias - treino')
axes[0].set_ylabel('Artigos')

pd.Series(dict(zip(TARGET_CATS, Ym_train.sum(axis=0)))).plot.bar(ax=axes[1])
axes[1].set_title('Frequencia multi-label - treino')
axes[1].set_ylabel('Rotulos positivos')
plt.tight_layout()
plt.show()


## Busca de C - categoria primaria


In [ ]:
from time import perf_counter

# A busca de hiperparametro usa uma subamostra reproduzivel. O modelo final,
# treinado mais abaixo, continua usando X_train completo.
TUNE_SIZE = min(12000, len(X_train))
rng_tune = np.random.default_rng(SEED)
idx_tune = rng_tune.choice(len(X_train), size=TUNE_SIZE, replace=False)
X_tune = X_train[idx_tune]
yp_tune = yp_train[idx_tune]
Ym_tune = Ym_train[idx_tune]

print(f'Busca de C usando {TUNE_SIZE} de {len(X_train)} exemplos de treino.')
resultados_primary = []
for C in C_GRID:
    inicio = perf_counter()
    model = make_pipeline(
        StandardScaler(),
        LinearSVC(C=C, dual=False, tol=1e-3, random_state=SEED, max_iter=5000),
    )
    model.fit(X_tune, yp_tune)
    pred_train = model.predict(X_tune)
    pred_val = model.predict(X_val)
    linha = {
        'C': C,
        'f1_macro_treino': f1_score(yp_tune, pred_train, average='macro'),
        'f1_macro_validacao': f1_score(yp_val, pred_val, average='macro'),
        'balanced_acc_validacao': balanced_accuracy_score(yp_val, pred_val),
        'segundos': perf_counter() - inicio,
    }
    resultados_primary.append(linha)
    print(f"C={C:<4} | F1 val={linha['f1_macro_validacao']:.4f} | {linha['segundos']:.1f}s")

primary_search = pd.DataFrame(resultados_primary)
display(primary_search)
BEST_C_PRIMARY = float(primary_search.sort_values(
    ['f1_macro_validacao', 'balanced_acc_validacao'], ascending=False
).iloc[0]['C'])
print('Melhor C primario:', BEST_C_PRIMARY)


## Busca de C - multi-label


In [ ]:
resultados_multi = []
for C in C_GRID:
    inicio = perf_counter()
    model = OneVsRestClassifier(
        make_pipeline(
            StandardScaler(),
            LinearSVC(C=C, dual=False, tol=1e-3, random_state=SEED, max_iter=5000),
        ),
        n_jobs=-1,
    )
    model.fit(X_tune, Ym_tune)
    pred_train = model.predict(X_tune)
    pred_val = model.predict(X_val)
    linha = {
        'C': C,
        'f1_macro_treino': f1_score(Ym_tune, pred_train, average='macro', zero_division=0),
        'f1_macro_validacao': f1_score(Ym_val, pred_val, average='macro', zero_division=0),
        'f1_micro_validacao': f1_score(Ym_val, pred_val, average='micro', zero_division=0),
        'segundos': perf_counter() - inicio,
    }
    resultados_multi.append(linha)
    print(f"C={C:<4} | F1 macro val={linha['f1_macro_validacao']:.4f} | {linha['segundos']:.1f}s")

multi_search = pd.DataFrame(resultados_multi)
display(multi_search)
BEST_C_MULTI = float(multi_search.sort_values(
    ['f1_macro_validacao', 'f1_micro_validacao'], ascending=False
).iloc[0]['C'])
print('Melhor C multi-label:', BEST_C_MULTI)


## Treino calibrado dos dois modelos


In [ ]:
print('Treinando modelo primario calibrado com todos os exemplos...')
primary_model = CalibratedClassifierCV(
    make_pipeline(
        StandardScaler(),
        LinearSVC(C=BEST_C_PRIMARY, dual=False, tol=1e-3, random_state=SEED, max_iter=5000),
    ),
    method='sigmoid', cv=3, n_jobs=-1,
)
primary_model.fit(X_train, yp_train)

print('Treinando modelo multi-label calibrado com todos os exemplos...')
multilabel_model = OneVsRestClassifier(
    CalibratedClassifierCV(
        make_pipeline(
            StandardScaler(),
            LinearSVC(C=BEST_C_MULTI, dual=False, tol=1e-3, random_state=SEED, max_iter=5000),
        ),
        method='sigmoid', cv=3, n_jobs=1,
    ),
    n_jobs=-1,
)
multilabel_model.fit(X_train, Ym_train)

primary_val_proba = primary_model.predict_proba(X_val)
multi_val_proba = multilabel_model.predict_proba(X_val)
print('Modelos calibrados treinados.')


## Threshold individual por categoria


In [ ]:
def melhor_threshold(y_true, scores):
    candidatos = np.linspace(0.05, 0.95, 181)
    f1s = [f1_score(y_true, scores >= t, zero_division=0) for t in candidatos]
    return float(candidatos[int(np.argmax(f1s))]), float(max(f1s))

thresholds = {}
threshold_rows = []
for i, classe in enumerate(TARGET_CATS):
    threshold, best_f1 = melhor_threshold(Ym_val[:, i], multi_val_proba[:, i])
    thresholds[classe] = threshold
    threshold_rows.append({'classe': classe, 'threshold': threshold, 'f1_validacao': best_f1})

threshold_table = pd.DataFrame(threshold_rows).sort_values('classe')
display(threshold_table)


## Metricas da categoria primaria


In [ ]:
primary_test_proba = primary_model.predict_proba(X_test)
primary_model_labels = np.asarray(primary_model.classes_, dtype=int)
primary_class_names = primary_encoder.inverse_transform(primary_model_labels)
primary_pred = primary_model_labels[np.argmax(primary_test_proba, axis=1)]

unseen_test = sorted(set(np.unique(yp_test)) - set(primary_model_labels))
if unseen_test:
    raise RuntimeError(
        'O teste possui categorias primarias ausentes no treino: '
        + ', '.join(primary_encoder.inverse_transform(unseen_test))
    )

def expected_calibration_error(y_true, proba, class_labels, n_bins=15):
    confidence = proba.max(axis=1)
    prediction = class_labels[proba.argmax(axis=1)]
    correct = (prediction == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

label_to_column = {label: column for column, label in enumerate(primary_model_labels)}
primary_target_matrix = np.zeros_like(primary_test_proba)
for row, label in enumerate(yp_test):
    primary_target_matrix[row, label_to_column[label]] = 1.0

primary_metrics = {
    'accuracy': accuracy_score(yp_test, primary_pred),
    'balanced_accuracy': balanced_accuracy_score(yp_test, primary_pred),
    'f1_macro': f1_score(yp_test, primary_pred, average='macro'),
    'f1_micro': f1_score(yp_test, primary_pred, average='micro'),
    'top3_accuracy': top_k_accuracy_score(
        yp_test, primary_test_proba, k=3, labels=primary_model_labels
    ),
    'log_loss': log_loss(yp_test, primary_test_proba, labels=primary_model_labels),
    'brier_multiclass': np.mean(np.sum((primary_test_proba - primary_target_matrix) ** 2, axis=1)),
    'ece_15_bins': expected_calibration_error(
        yp_test, primary_test_proba, primary_model_labels
    ),
}
display(pd.Series(primary_metrics, name='categoria_primaria'))
print('Classes primarias aprendidas:', list(primary_class_names))
print('Classes alvo sem exemplos primarios:', sorted(set(TARGET_CATS) - set(primary_class_names)))
print(classification_report(
    yp_test,
    primary_pred,
    labels=primary_model_labels,
    target_names=primary_class_names,
    zero_division=0,
))


## Matriz de confusao primaria


In [ ]:
cm = confusion_matrix(yp_test, primary_pred, labels=primary_model_labels)
plt.figure(figsize=(13, 11))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=primary_class_names,
    yticklabels=primary_class_names,
)
plt.title('Matriz de confusao - categoria primaria')
plt.xlabel('Predita')
plt.ylabel('Real')
plt.tight_layout()
plt.show()


## Metricas multi-label


In [ ]:
multi_test_proba = multilabel_model.predict_proba(X_test)
threshold_vector = np.array([thresholds[c] for c in TARGET_CATS])
multi_pred = (multi_test_proba >= threshold_vector).astype(int)

# A categoria primaria prevista sempre pertence ao conjunto final retornado.
for row, primary_idx in enumerate(primary_pred):
    primary_name = primary_encoder.inverse_transform([primary_idx])[0]
    multi_pred[row, TARGET_CATS.index(primary_name)] = 1

multi_metrics = {
    'f1_micro': f1_score(Ym_test, multi_pred, average='micro', zero_division=0),
    'f1_macro': f1_score(Ym_test, multi_pred, average='macro', zero_division=0),
    'precision_micro': precision_score(Ym_test, multi_pred, average='micro', zero_division=0),
    'precision_macro': precision_score(Ym_test, multi_pred, average='macro', zero_division=0),
    'recall_micro': recall_score(Ym_test, multi_pred, average='micro', zero_division=0),
    'recall_macro': recall_score(Ym_test, multi_pred, average='macro', zero_division=0),
    'hamming_loss': hamming_loss(Ym_test, multi_pred),
    'subset_accuracy': accuracy_score(Ym_test, multi_pred),
    'lrap': label_ranking_average_precision_score(Ym_test, multi_test_proba),
    'coverage_error': coverage_error(Ym_test, multi_test_proba),
}
display(pd.Series(multi_metrics, name='multi_label'))

per_class = []
for i, classe in enumerate(TARGET_CATS):
    p, r, f1, support = precision_recall_fscore_support(Ym_test[:, i], multi_pred[:, i], average='binary', zero_division=0)
    per_class.append({
        'classe': classe, 'precision': p, 'recall': r, 'f1': f1,
        'support': int(Ym_test[:, i].sum()),
        'pr_auc': average_precision_score(Ym_test[:, i], multi_test_proba[:, i]),
        'threshold': thresholds[classe],
        'brier': brier_score_loss(Ym_test[:, i], multi_test_proba[:, i]),
    })
per_class_df = pd.DataFrame(per_class).sort_values('f1')
display(per_class_df)


## Diagnostico de classes proximas e calibracao


In [ ]:
FOCUS = ['cs.LG', 'cs.AI', 'cs.CL', 'math-ph', 'math.MP']
fig, axes = plt.subplots(len(FOCUS), 2, figsize=(13, 4 * len(FOCUS)))
for row, classe in enumerate(FOCUS):
    i = TARGET_CATS.index(classe)
    axes[row, 0].hist(multi_test_proba[Ym_test[:, i] == 0, i], bins=25, alpha=.65, label='real=0')
    axes[row, 0].hist(multi_test_proba[Ym_test[:, i] == 1, i], bins=25, alpha=.65, label='real=1')
    axes[row, 0].axvline(thresholds[classe], color='black', linestyle='--', label='threshold')
    axes[row, 0].set_title(f'Scores - {classe}')
    axes[row, 0].legend()

    frac_pos, mean_pred = calibration_curve(Ym_test[:, i], multi_test_proba[:, i], n_bins=10, strategy='quantile')
    axes[row, 1].plot(mean_pred, frac_pos, marker='o')
    axes[row, 1].plot([0, 1], [0, 1], '--', color='gray')
    axes[row, 1].set_title(f'Calibracao - {classe}')
    axes[row, 1].set_xlabel('Probabilidade prevista')
    axes[row, 1].set_ylabel('Frequencia observada')
plt.tight_layout()
plt.show()


## Exportacao para o backend


## Comparacao com o experimento de 10.500 artigos

A comparacao abaixo usa apenas metricas primarias que existiam no experimento anterior. Metricas multi-label de 10.500 nao existem porque aquela pipeline reduzia cada paper a um unico `assigned_category`; elas nao devem ser inventadas nem comparadas diretamente.


In [ ]:
baseline_10500 = {
    'accuracy': 0.699683,
    'balanced_accuracy': 0.699683,
    'f1_macro': 0.697704,
}
comparacao = pd.DataFrame([
    {'experimento': '10500_single_label', **baseline_10500},
    {'experimento': '40000_primary_calibrado', **{k: primary_metrics[k] for k in baseline_10500}},
]).set_index('experimento')
display(comparacao)
display((comparacao.loc['40000_primary_calibrado'] - comparacao.loc['10500_single_label']).rename('ganho_absoluto'))


In [ ]:
ARTEFATO = PIPE / 'arxiv_modelos_40000_calibrados_multilabel.joblib'
METRICAS = PIPE / 'arxiv_metricas_40000_calibradas_multilabel.json'

bundle = {
    'version': 1,
    'primary_model': primary_model,
    'multilabel_model': multilabel_model,
    'primary_classes': list(primary_class_names),
    'multilabel_classes': list(TARGET_CATS),
    'thresholds': thresholds,
    'best_c_primary': BEST_C_PRIMARY,
    'best_c_multilabel': BEST_C_MULTI,
    'embedding_model': 'sentence-transformers/allenai-specter',
    'input_format': 'title [SEP] abstract_reduzido',
    'dataset_file': NOME_JSONL,
    'sample_size': int(len(df)),
    'split_seed': SEED,
}
joblib.dump(bundle, ARTEFATO, compress=3)

metrics_payload = {
    'primary': {k: float(v) for k, v in primary_metrics.items()},
    'multilabel': {k: float(v) for k, v in multi_metrics.items()},
    'per_class': per_class_df.to_dict(orient='records'),
    'thresholds': thresholds,
}
METRICAS.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

print('Modelo:', ARTEFATO)
print('Metricas:', METRICAS)
print('Copie o JSONL e o JOBLIB para backend/data para ativar o modo 40k.')
